In [3]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
import os
# Mount Drive and Define Paths
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/value-vortex/'
DATA_PATH = PROJECT_PATH + 'dataset/'
FEATURES_PATH = PROJECT_PATH + 'features/'
os.makedirs(FEATURES_PATH, exist_ok=True)


# Load Data
train_df = pd.read_csv(DATA_PATH + 'train.csv')
test_df = pd.read_csv(DATA_PATH + 'test.csv')

print("Setup Complete. Data is loaded.")

Mounted at /content/drive
Setup Complete. Data is loaded.


In [4]:
# --- Extraction Logic ---
def extract_ipq(text):
    # This pattern looks for "Item Pack Quantity:", optional spaces, and then captures the number(s)
    match = re.search(r"Item Pack Quantity:\s*(\d+)", str(text), re.IGNORECASE)
    if match:
        return int(match.group(1))
    else:
        return np.nan # Return NaN if the pattern is not found

print("Extracting IPQ from training set...")
train_df['ipq'] = train_df['catalog_content'].apply(extract_ipq)

print("Extracting IPQ from test set...")
test_df['ipq'] = test_df['catalog_content'].apply(extract_ipq)

# --- Handle Missing Values ---
# It's a reasonable assumption that if IPQ is not mentioned, the item is a single pack.
train_df['ipq'].fillna(1, inplace=True)
test_df['ipq'].fillna(1, inplace=True)
print("\nMissing IPQ values filled with 1.")

# Let's check the result
print("\nDistribution of IPQ in the training set:")
print(train_df['ipq'].value_counts().head())

# --- Save the Deliverable ---
np.save(FEATURES_PATH + 'ipq_train.npy', train_df['ipq'].values)
np.save(FEATURES_PATH + 'ipq_test.npy', test_df['ipq'].values)

print(f"\nSUCCESS: IPQ features saved to {FEATURES_PATH}")

Extracting IPQ from training set...
Extracting IPQ from test set...


/tmp/ipython-input-557846973.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['ipq'].fillna(1, inplace=True)
/tmp/ipython-input-557846973.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.m


Missing IPQ values filled with 1.

Distribution of IPQ in the training set:
ipq
1.0    75000
Name: count, dtype: int64

SUCCESS: IPQ features saved to /content/drive/MyDrive/value-vortex/features/


In [5]:
!pip install sentence-transformers -q

In [6]:
from sentence_transformers import SentenceTransformer

# 1. Load the pre-trained model
print("Loading sentence transformer model...")
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
print("Model loaded.")

# 2. Combine text from both train and test sets for efficient processing
all_text = pd.concat([train_df['catalog_content'].astype(str), test_df['catalog_content'].astype(str)]).tolist()

# 3. Generate embeddings (this will take 15-30 mins on a GPU)
print(f"Generating embeddings for {len(all_text)} text entries...")
embeddings = model.encode(all_text, show_progress_bar=True, device='cuda')
print("Embeddings generated.")

# 4. Split embeddings back into train and test sets
train_embeddings = embeddings[:len(train_df)]
test_embeddings = embeddings[len(train_df):]

# 5. Save the Deliverables
np.save(FEATURES_PATH + 'train_embeddings_minilm.npy', train_embeddings)
np.save(FEATURES_PATH + 'test_embeddings_minilm.npy', test_embeddings)

print(f"\nSUCCESS: Sentence embeddings saved to {FEATURES_PATH}")
print("Train embeddings shape:", train_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

Loading sentence transformer model...
Model loaded.
Generating embeddings for 150000 text entries...


Batches:   0%|          | 0/4688 [00:00<?, ?it/s]

Embeddings generated.

SUCCESS: Sentence embeddings saved to /content/drive/MyDrive/value-vortex/features/
Train embeddings shape: (75000, 384)
Test embeddings shape: (75000, 384)
